[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5509Files/blob/main/OPIM5509_Module3_Files/notebooks/What_Autoencoders_Can_Actually_Do.ipynb)

# What can you actually DO with an autoencoder?

We already watched an autoencoder squish an image down to a tiny 32-number code and rebuild it. Cute trick - but *so what?* Turns out that little code (and the reconstruction error that comes with it) buys you four genuinely useful things:

1. **Catch outliers** - flag data the model has never seen before.
2. **Free features** - train a classifier on the 32-number code instead of 784 pixels.
3. **Clean up noise** - turn the same idea into an image denoiser.
4. **Find structure** - watch the digits cluster themselves, with no labels at all.

Everything below runs on MNIST, off of one little autoencoder.

## First, train the autoencoder

Deep dense autoencoder: 784 pixels squished to a **32-number code**, then rebuilt. We hang onto an `encoder` handle so we can grab that code later.

In [ ]:
from keras.datasets import mnist, fashion_mnist
from keras.layers import Input, Dense
from keras.models import Model
from keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
import keras
keras.utils.set_random_seed(5509)

(x_train, y_train), (x_test, y_test) = mnist.load_data()
X_train = (x_train.astype("float32") / 255.).reshape(len(x_train), 784)
X_test  = (x_test.astype("float32")  / 255.).reshape(len(x_test), 784)

# 784 -> 128 -> 64 -> 32 (the code) -> 64 -> 128 -> 784
inp  = Input(shape=(784,))
e    = Dense(128, activation="relu")(inp)
e    = Dense(64,  activation="relu")(e)
code = Dense(32,  activation="relu")(e)        # <- the tiny latent code
d    = Dense(64,  activation="relu")(code)
d    = Dense(128, activation="relu")(d)
out  = Dense(784, activation="sigmoid")(d)

autoencoder = Model(inp, out)
encoder     = Model(inp, code)                 # same weights, stops at the code
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")

es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)
history = autoencoder.fit(X_train, X_train,              # input and target are both the image
                epochs=100, batch_size=256, shuffle=True,
                validation_data=(X_test, X_test), callbacks=[es])

In [ ]:
# did it train? the loss curves say yes
plt.plot(history.history['loss'],     label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('epoch'); plt.ylabel('loss (binary crossentropy)'); plt.legend()
plt.title('Autoencoder training'); plt.show()

## Use case 1: catch outliers (anomaly detection)

The autoencoder only ever learned to rebuild **digits**. Hand it something from a different world - a Fashion-MNIST sneaker - and it makes a mess of the reconstruction. So the **reconstruction error is an anomaly score**: small for things it understands, big for things it doesn't. No labels required.

In [ ]:
def recon_error(X):
    """Per-image reconstruction error (mean squared pixel error)."""
    recon = autoencoder.predict(X, verbose=0)
    return np.mean((recon - X) ** 2, axis=1)

err_digits = recon_error(X_test)                     # things it knows

(_, _), (fashion, _) = fashion_mnist.load_data()     # things it has NEVER seen
X_fashion = (fashion.astype("float32") / 255.).reshape(len(fashion), 784)
err_fashion = recon_error(X_fashion)

# Pick a threshold from the NORMAL data only: 95th percentile of digit error.
threshold = np.percentile(err_digits, 95)
print(f"threshold (95th pct of digit error): {threshold:.4f}")
print(f"flagged as outliers -> Fashion: {100*np.mean(err_fashion > threshold):.0f}%   digits: {100*np.mean(err_digits > threshold):.0f}%")

In [ ]:
# the two error distributions barely overlap
plt.figure(figsize=(7,4))
plt.hist(err_digits,  bins=60, alpha=0.7, label="digits (normal)")
plt.hist(err_fashion, bins=60, alpha=0.7, label="Fashion (outliers)")
plt.axvline(threshold, color="k", ls="--", label="threshold")
plt.xlabel("reconstruction error"); plt.ylabel("count"); plt.legend()
plt.title("Outliers reconstruct badly -> high error"); plt.show()

In [ ]:
# see it: a normal digit rebuilds fine; a sneaker comes out garbled
def show_pair(x_flat, title):
    r = autoencoder.predict(x_flat[None, :], verbose=0)[0]
    fig, ax = plt.subplots(1, 2, figsize=(3.2, 1.9))
    ax[0].imshow(x_flat.reshape(28,28), cmap="gray"); ax[0].set_title("in");  ax[0].axis("off")
    ax[1].imshow(r.reshape(28,28),      cmap="gray"); ax[1].set_title("rebuilt"); ax[1].axis("off")
    plt.suptitle(title, fontsize=10); plt.show()

show_pair(X_test[0],    f"digit  (error {err_digits[0]:.3f})")
show_pair(X_fashion[0], f"sneaker (error {err_fashion[0]:.3f}) - outlier!")

## Use case 2: free features (the 32-number code as classifier input)

That 32-number code is a compressed summary of the whole image. So feed **the code** into a small classifier instead of the 784 raw pixels - **24x fewer inputs** - and it still nails the digit. The autoencoder did the feature engineering for us, and it never saw a single label to do it.

In [ ]:
codes_train = encoder.predict(X_train, verbose=0)   # (60000, 32)
codes_test  = encoder.predict(X_test,  verbose=0)   # (10000, 32)

clf = keras.Sequential([
    Input(shape=(32,)),                 # <- just the 32-number code
    Dense(64, activation="relu"),
    Dense(10, activation="softmax"),
])
clf.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["acc"])
clf.fit(codes_train, y_train, epochs=30, batch_size=256,
        validation_data=(codes_test, y_test),
        callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)])

acc = clf.evaluate(codes_test, y_test, verbose=0)[1]
print(f"\ndigit accuracy from a 32-number code: {acc:.3f}   (vs 784 raw pixels)")

## Use case 3: clean up noisy images (denoising)

An autoencoder doesn't have to rebuild *itself*. Train it to turn a **dirty** image into a **clean** one and you've got a denoiser.

We make the dirt ourselves - and we're deliberate about **leakage**: we add noise to the train and test sets **independently**, fit the denoiser **only** on the training pairs (`noisy_train -> clean_train`), and never let it touch the test images or their noise during training. The "clean" targets are simply the original images.

In [ ]:
rng = np.random.default_rng(0)
noise = 0.5

# our own dirty data - train and test noised SEPARATELY (independent noise)
X_train_noisy = np.clip(X_train + noise * rng.normal(size=X_train.shape), 0, 1).astype("float32")
X_test_noisy  = np.clip(X_test  + noise * rng.normal(size=X_test.shape),  0, 1).astype("float32")

# a denoiser: learns dirty -> clean. Note the input is NOISY, the target is CLEAN.
di = Input(shape=(784,))
h  = Dense(128, activation="relu")(di)
h  = Dense(64,  activation="relu")(h)
h  = Dense(128, activation="relu")(h)
do = Dense(784, activation="sigmoid")(h)
denoiser = Model(di, do)
denoiser.compile(optimizer="adam", loss="binary_crossentropy")

denoiser.fit(X_train_noisy, X_train,                 # FIT ON TRAIN ONLY: noisy in, clean out
             epochs=100, batch_size=256, shuffle=True,
             validation_data=(X_test_noisy, X_test), # test held out, noised independently
             callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1)])

In [ ]:
# on the held-out TEST set: clean original / our noisy version / the denoiser's cleanup
cleaned = denoiser.predict(X_test_noisy[:8], verbose=0)
rows = [("original", X_test), ("we added noise", X_test_noisy), ("denoised", cleaned)]
plt.figure(figsize=(12, 4.5))
for r, (label, data) in enumerate(rows):
    for i in range(8):
        ax = plt.subplot(3, 8, r*8 + i + 1)
        ax.imshow(data[i].reshape(28, 28), cmap="gray"); ax.axis("off")
        if i == 0: ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=11); ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Denoiser: dirty in -> clean out (test images it never trained on)", fontsize=13)
plt.tight_layout(); plt.show()

## Use case 4: find structure (cluster the codes)

We never handed the autoencoder any labels - it only learned to compress. But take those 32-number codes, squash them to 2D with t-SNE, and color by the true digit: the digits have **sorted themselves into groups**. The structure was always in the data; compressing it surfaced it. (We use the labels *only* to color the dots - never to train.)

In [ ]:
from sklearn.manifold import TSNE

sample = 3000
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(codes_test[:sample])

plt.figure(figsize=(7, 6))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=y_test[:sample], cmap="tab10", s=8)
plt.colorbar(sc, label="true digit (coloring only!)")
plt.title("32-number codes -> t-SNE 2D: the digits cluster themselves"); plt.show()

## Takeaways

One little autoencoder, four payoffs:

- **Reconstruction error is a built-in outlier detector** - big error means "I've never seen anything like this."
- **The latent code is free, compact features** - unsupervised feature engineering you can drop into any classifier.
- **Swap the target from "itself" to "the clean version"** and the same architecture becomes a denoiser.
- **The code space has structure** - similar inputs land near each other, so clustering just works.

That's why the "squish it into a nugget" trick is worth caring about - the nugget is useful.